# Cheap Reset vs Smart Cache Memory on Real Traffic\n\nThis notebook demonstrates a W-TinyLFU cache-admission simulator that compares two ways of tracking key popularity for cache admission decisions:\n\n- **Baseline (`GlobalResetFrequencyEstimator`)**: a single Count-Min sketch that is reset (halved) wholesale on a fixed schedule (Caffeine's approach).\n- **Proposed (`PerKeyDecayFrequencyEstimator`)**: three tiered Count-Min sketches with different halving periods; each key is assigned to a tier based on the coefficient of variation (CoV) of its inter-arrival gaps (bursty keys get a short half-life, regular keys a long half-life).\n\nThe demo has two parts, both minimal-scale reproductions of the original experiment's logic:\n\n- **Part A**: sweeps very short global-reset multipliers (1x/2x/4x cache capacity) on a synthetic Zipf trace with injected \"drift\" (hot keys change identity), and compares the best short-reset baseline's recovery time against the proposed per-key-decay estimator's recovery time (the latter loaded from iter1's precomputed results, not rerun).\n- **Part B**: replays both estimators over a real sample of the Twitter production cache trace (`twitter/cache-trace`, cluster026), and runs a lightweight unsupervised JS-divergence changepoint detector, first validated on synthetic data with known drift events.\n\nAll core simulator code (sketches, doorkeeper, SLRU, estimators, trace generator, recovery-time metric) is reused **unchanged** from iter1's `iter1_method.py`, exactly as the original script imports it.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# loguru is NOT pre-installed on Colab -- install unconditionally\n_pip('loguru==0.7.3')\n\n# numpy, matplotlib are pre-installed on Colab; install locally only, at Colab's exact versions\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations\n\nimport gc\nimport json\nimport sys\nimport time\nfrom collections import Counter, OrderedDict\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom loguru import logger\n\nlogger.remove()\nlogger.add(sys.stdout, level=\"INFO\", format=\"{time:HH:mm:ss}|{level:<7}|{message}\")

In [ ]:
GITHUB_DATA_URL = \"https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-b940ce-shadow-queue-admission-with-recency/main/round-2/experiment-1/demo/mini_demo_data.json\"\nimport os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception:\n        pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f:\n            return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
data = load_data()\nreal_trace_sample = data[\"real_trace_sample\"]  # ~3000-request sample of the real Twitter cluster026 trace\niter1_proposed_recovery_times = data[\"iter1_proposed_recovery_times\"]  # iter1's precomputed proposed-estimator results\nprint(f\"Loaded {len(real_trace_sample)} real-trace requests and {len(iter1_proposed_recovery_times)} iter1 proposed-estimator rows\")

## Configuration\n\nAll tunable parameters are collected here. Values are set to the **minimum scale that still produces meaningful output** for a fast demo run; the original (full-scale) values are noted in comments. Increase `KEY_SPACE` / `N_REQUESTS_MAIN` / `SEEDS` toward the commented-out originals for a closer (but much slower) reproduction.

In [ ]:
# --- Part A (synthetic drift sweep) config ---\nRATIO = 0.05             # original: 0.01\nALPHA = 1.2               # unchanged from original (win-corner cell)\nKEY_SPACE = 3000          # original: 150_000\nCACHE_CAPACITY = max(10, int(RATIO * KEY_SPACE))  # original formula unchanged; = 150 at these settings (was 1500)\nSHORT_MULTIPLIERS = [1, 2, 4]  # unchanged from original\nSEEDS = [1, 2]             # original: [1, 2, 3]\nN_REQUESTS_MAIN = 60_000  # original: 600_000\nRECOVERY_LOOKAHEAD_MAIN = 6_000  # original: 60_000\nBURST_PROB = 0.5           # unchanged from original\n\nDRIFT_SCENARIOS = [\n    {\"name\": \"low_mag_low_freq\", \"drift_magnitude\": 0.05, \"n_drift_events\": 2},\n    {\"name\": \"low_mag_high_freq\", \"drift_magnitude\": 0.05, \"n_drift_events\": 8},\n    {\"name\": \"high_mag_low_freq\", \"drift_magnitude\": 0.20, \"n_drift_events\": 2},\n    {\"name\": \"high_mag_high_freq\", \"drift_magnitude\": 0.20, \"n_drift_events\": 8},\n]  # unchanged from original (identical to iter1's DRIFT_SCENARIOS)\n\nSAMPLE_MULTIPLIERS = [4, 8, 16, 32]  # iter1's already-swept multipliers, unchanged\nSHADOW_QUEUE_MULT = 2                # unchanged from original\nIT1_BEST_MULTIPLIER_AT_CELL = 32     # iter1's Phase A tuning result at this cell, unchanged\n\n# --- Part B (real trace + changepoint detector) config ---\nCP_WINDOW = 2000       # unchanged from original\nCP_STRIDE = 500        # unchanged from original\nCP_TOP_K = 50           # unchanged from original\nCP_PERCENTILE = 95.0    # unchanged from original\nCP_RECOVERY_LOOKAHEAD = 5000  # unchanged from original\n\nprint(f\"cache_capacity={CACHE_CAPACITY}, key_space={KEY_SPACE}, n_requests_main={N_REQUESTS_MAIN}, seeds={SEEDS}\")

## Base simulator (iter1, imported unchanged)\n\nThe original `method.py` does `import iter1_method as base` and reuses its classes/functions completely unchanged. Since a notebook can't `import` a sibling `.py` file on Colab, the cell below inlines iter1's simulator code **verbatim** (same classes, same logic, same constants/thresholds) so it plays the exact role of `base.*` in the rest of this notebook: the Count-Min sketch, doorkeeper, both frequency estimators, the SLRU + W-TinyLFU admission cache, the synthetic Zipf-drift trace generator, and the recovery-time metric.

In [ ]:
RNG_SEED_SALT = 0x9E3779B1  # fixed odd constant for deterministic integer hashing\n\n\nclass CountMin4Bit:\n    \"\"\"Depth-4 Count-Min sketch with 4-bit saturating counters, 2 per byte.\n\n    Matches Caffeine's `FrequencySketch`: increment saturates at 15, estimate\n    is the min across rows, and `halve_all` implements the RESET_MASK trick\n    (right-shift each nibble by 1, in place, in a single pass over bytes).\n    \"\"\"\n\n    DEPTH = 4\n    _RESET_MASK = 0x77  # 0111_0111: halves both nibbles, drops each LSB\n\n    def __init__(self, num_counters: int, seed: int):\n        self.width = max(16, num_counters | 1)  # odd width reduces hash collisions across rows\n        self.table = bytearray((self.width + 1) // 2)\n        rng = np.random.default_rng(seed ^ RNG_SEED_SALT)\n        # odd multipliers for a simple deterministic multiplicative hash per row\n        self._salts = [int(x) | 1 for x in rng.integers(1, 2**31 - 1, size=self.DEPTH)]\n\n    def _pos(self, key: int, row: int) -> int:\n        return ((key ^ self._salts[row]) * self._salts[(row + 1) % self.DEPTH]) % self.width\n\n    def _get_nibble(self, pos: int) -> int:\n        b = self.table[pos >> 1]\n        return b & 0x0F if pos & 1 == 0 else (b >> 4) & 0x0F\n\n    def _set_nibble(self, pos: int, value: int) -> None:\n        idx = pos >> 1\n        b = self.table[idx]\n        if pos & 1 == 0:\n            self.table[idx] = (b & 0xF0) | value\n        else:\n            self.table[idx] = (b & 0x0F) | (value << 4)\n\n    def increment(self, key: int) -> None:\n        for row in range(self.DEPTH):\n            pos = self._pos(key, row)\n            v = self._get_nibble(pos)\n            if v < 15:\n                self._set_nibble(pos, v + 1)\n\n    def estimate(self, key: int) -> int:\n        return min(self._get_nibble(self._pos(key, row)) for row in range(self.DEPTH))\n\n    def halve_all(self) -> None:\n        table = self.table\n        mask = self._RESET_MASK\n        for i in range(len(table)):\n            table[i] = (table[i] >> 1) & mask\n\n    def memory_bytes(self) -> int:\n        return len(self.table) + self.DEPTH * 8  # counters + salts\n\n\nclass Doorkeeper:\n    \"\"\"1-bit-per-slot Bloom-style first-touch filter, cleared with the sketch.\"\"\"\n\n    def __init__(self, num_bits: int, seed: int):\n        self.num_bits = max(16, num_bits | 1)\n        self.bits = bytearray((self.num_bits + 7) // 8)\n        rng = np.random.default_rng((seed ^ 0xD1B54A35) & 0x7FFFFFFF)\n        self._salt = int(rng.integers(1, 2**31 - 1)) | 1\n\n    def _pos(self, key: int) -> int:\n        return ((key ^ self._salt) * 2654435761) % self.num_bits\n\n    def contains(self, key: int) -> bool:\n        pos = self._pos(key)\n        return bool(self.bits[pos >> 3] & (1 << (pos & 7)))\n\n    def maybe_add(self, key: int) -> bool:\n        \"\"\"Returns True iff the key was NOT already present (first touch).\"\"\"\n        pos = self._pos(key)\n        byte_idx, bit = pos >> 3, 1 << (pos & 7)\n        if self.bits[byte_idx] & bit:\n            return False\n        self.bits[byte_idx] |= bit\n        return True\n\n    def clear(self) -> None:\n        for i in range(len(self.bits)):\n            self.bits[i] = 0\n\n    def memory_bytes(self) -> int:\n        return len(self.bits) + 8\n\n\nclass GlobalResetFrequencyEstimator:\n    \"\"\"Baseline: single Count-Min sketch, reset (halved) globally on a schedule.\"\"\"\n\n    name = \"global_reset_tinylfu\"\n\n    def __init__(self, cache_capacity: int, sample_size_multiplier: int, seed: int):\n        self.sketch = CountMin4Bit(4 * cache_capacity, seed=seed)\n        self.doorkeeper = Doorkeeper(cache_capacity * 8, seed=seed + 1)\n        self.sample_size = max(1, sample_size_multiplier * cache_capacity)\n        self.size = 0\n        self.sample_size_multiplier = sample_size_multiplier\n\n    def record_access(self, key: int) -> None:\n        if not self.doorkeeper.maybe_add(key):\n            self.sketch.increment(key)\n        self.size += 1\n        if self.size >= self.sample_size:\n            self.sketch.halve_all()\n            self.doorkeeper.clear()\n            self.size = 0\n\n    def frequency(self, key: int) -> int:\n        return self.sketch.estimate(key) + (1 if self.doorkeeper.contains(key) else 0)\n\n    def memory_bytes(self) -> int:\n        return self.sketch.memory_bytes() + self.doorkeeper.memory_bytes()\n\n\nclass _LRUMeta:\n    \"\"\"Bounded LRU dict for per-key shadow metadata (read-peek vs touch-on-write).\"\"\"\n\n    def __init__(self, capacity: int):\n        self.capacity = max(1, capacity)\n        self._od: \"OrderedDict[int, tuple]\" = OrderedDict()\n\n    def peek(self, key: int):\n        return self._od.get(key)\n\n    def put_and_touch(self, key: int, value: tuple) -> None:\n        if key in self._od:\n            self._od.move_to_end(key)\n        self._od[key] = value\n        if len(self._od) > self.capacity:\n            self._od.popitem(last=False)\n\n    def __len__(self) -> int:\n        return len(self._od)\n\n    def memory_bytes(self) -> int:\n        # 5-field tuple of Python numbers + dict/OrderedDict per-entry overhead;\n        # ~120 bytes/entry is a conservative empirical estimate for this shape.\n        return len(self._od) * 120 + 200\n\n\n# CoV thresholds for the 3-tier classifier. CoV==1 is the memoryless\n# (Poisson/exponential) reference point: renewal processes with CoV well\n# above 1 are bursty (many small gaps + occasional huge gaps -> volatile,\n# short half-life is right), well below 1 are near-regular/periodic\n# (long half-life is right, since the popularity signal is stable).\nCOV_HIGH_THRESH = 1.5\nCOV_LOW_THRESH = 0.5\nEWMA_ALPHA = 0.3\nMIN_OBS_FOR_CLASSIFICATION = 3\n\n\nclass PerKeyDecayFrequencyEstimator:\n    \"\"\"Proposed: K tiered Count-Min sketches, each with its own halving period.\n\n    Only keys currently tracked in a bounded shadow-metadata LRU get a\n    per-key inter-arrival CoV estimate and tier assignment; a key that falls\n    out of the shadow queue reverts to the default tier on re-entry, bounding\n    memory at O(shadow_queue_capacity) regardless of the true key space.\n    \"\"\"\n\n    name = \"per_key_decay_tinylfu\"\n    TIERS = [(2, \"volatile\"), (8, \"default\"), (32, \"stable\")]\n    DEFAULT_TIER = 1\n\n    def __init__(self, cache_capacity: int, shadow_queue_capacity: int, seed: int):\n        self.tier_sketches = [\n            CountMin4Bit(4 * cache_capacity, seed=seed + 100 + t) for t in range(len(self.TIERS))\n        ]\n        self.tier_sample_size = [max(1, m * cache_capacity) for m, _ in self.TIERS]\n        self.tier_size = [0] * len(self.TIERS)\n        self.doorkeeper = Doorkeeper(cache_capacity * 8, seed=seed + 1)\n        self.shadow_meta = _LRUMeta(shadow_queue_capacity)\n        self.global_clock = 0\n        self.tier_assignment_counts = [0] * len(self.TIERS)  # diagnostics\n\n    def _classify(self, ewma_gap: float, ewma_gap_sq: float, n_obs: int) -> int:\n        if n_obs < MIN_OBS_FOR_CLASSIFICATION:\n            return self.DEFAULT_TIER\n        var = max(ewma_gap_sq - ewma_gap * ewma_gap, 0.0)\n        cov = (var**0.5) / max(ewma_gap, 1e-6)\n        if cov > COV_HIGH_THRESH:\n            return 0  # volatile / bursty\n        if cov < COV_LOW_THRESH:\n            return 2  # stable / regular\n        return 1  # default\n\n    def record_access(self, key: int) -> None:\n        self.global_clock += 1\n        meta = self.shadow_meta.peek(key)\n        if meta is None:\n            tier = self.DEFAULT_TIER\n            self.shadow_meta.put_and_touch(key, (self.global_clock, 0.0, 0.0, tier, 1))\n        else:\n            last_ts, ewma_gap, ewma_gap_sq, _prev_tier, n_obs = meta\n            gap = float(self.global_clock - last_ts)\n            if n_obs > 0:\n                ewma_gap = EWMA_ALPHA * gap + (1 - EWMA_ALPHA) * ewma_gap\n                ewma_gap_sq = EWMA_ALPHA * (gap * gap) + (1 - EWMA_ALPHA) * ewma_gap_sq\n            else:\n                ewma_gap, ewma_gap_sq = gap, gap * gap\n            n_obs += 1\n            tier = self._classify(ewma_gap, ewma_gap_sq, n_obs)\n            self.shadow_meta.put_and_touch(key, (self.global_clock, ewma_gap, ewma_gap_sq, tier, n_obs))\n\n        self.tier_assignment_counts[tier] += 1\n        if not self.doorkeeper.maybe_add(key):\n            self.tier_sketches[tier].increment(key)\n            self.tier_size[tier] += 1\n            if self.tier_size[tier] >= self.tier_sample_size[tier]:\n                self.tier_sketches[tier].halve_all()\n                self.tier_size[tier] = 0\n\n    def frequency(self, key: int) -> int:\n        meta = self.shadow_meta.peek(key)\n        tier = meta[3] if meta is not None else self.DEFAULT_TIER\n        base = self.tier_sketches[tier].estimate(key)\n        return base + (1 if self.doorkeeper.contains(key) else 0)\n\n    def memory_bytes(self) -> int:\n        return (\n            sum(s.memory_bytes() for s in self.tier_sketches)\n            + self.doorkeeper.memory_bytes()\n            + self.shadow_meta.memory_bytes()\n        )\n\n\nclass SLRUCache:\n    \"\"\"Segmented LRU: 80% protected / 20% probationary (Caffeine's default split).\"\"\"\n\n    def __init__(self, capacity: int):\n        self.capacity = max(1, capacity)\n        self.protected_capacity = max(1, int(0.8 * self.capacity))\n        self.probationary_capacity = max(1, self.capacity - self.protected_capacity)\n        self.protected: \"OrderedDict[int, None]\" = OrderedDict()\n        self.probationary: \"OrderedDict[int, None]\" = OrderedDict()\n\n    def get(self, key: int) -> bool:\n        if key in self.protected:\n            self.protected.move_to_end(key)\n            return True\n        if key in self.probationary:\n            del self.probationary[key]\n            self.protected[key] = None\n            if len(self.protected) > self.protected_capacity:\n                demoted, _ = self.protected.popitem(last=False)\n                self.probationary[demoted] = None\n                if len(self.probationary) > self.probationary_capacity:\n                    self.probationary.popitem(last=False)\n            return True\n        return False\n\n    def victim_for_admission_test(self) -> Optional[int]:\n        if self.probationary:\n            return next(iter(self.probationary))\n        return None\n\n    def admit_candidate(self, key: int) -> Optional[int]:\n        \"\"\"Admits into probationary MRU; evicts+returns probationary LRU if full.\"\"\"\n        evicted = None\n        if len(self.probationary) >= self.probationary_capacity and self.probationary:\n            evicted, _ = self.probationary.popitem(last=False)\n        self.probationary[key] = None\n        return evicted\n\n    def memory_bytes(self) -> int:\n        return (len(self.protected) + len(self.probationary)) * 56  # int key + OrderedDict entry overhead\n\n\nclass WindowTinyLFUCache:\n    \"\"\"Full W-TinyLFU: small LRU admission window + doorkeeper/sketch-gated SLRU main.\"\"\"\n\n    def __init__(self, capacity: int, estimator, window_frac: float = 0.01):\n        self.window_capacity = max(1, int(round(window_frac * capacity)))\n        self.main_capacity = max(1, capacity - self.window_capacity)\n        self.window: \"OrderedDict[int, None]\" = OrderedDict()\n        self.main = SLRUCache(self.main_capacity)\n        self.estimator = estimator\n\n    def access(self, key: int) -> bool:\n        \"\"\"Records the access with the estimator and returns True on a cache hit.\"\"\"\n        self.estimator.record_access(key)\n        if key in self.window:\n            self.window.move_to_end(key)\n            return True\n        if self.main.get(key):\n            return True\n        # miss: admit into the window; if the window overflows, its evicted\n        # LRU item competes for a main-region slot against the SLRU victim.\n        self.window[key] = None\n        if len(self.window) > self.window_capacity:\n            candidate, _ = self.window.popitem(last=False)\n            victim = self.main.victim_for_admission_test()\n            if victim is None or self.estimator.frequency(candidate) > self.estimator.frequency(victim):\n                self.main.admit_candidate(candidate)\n        return False\n\n    def memory_bytes(self) -> int:\n        return self.estimator.memory_bytes() + self.main.memory_bytes() + len(self.window) * 56\n\n\n@dataclass\nclass TraceResult:\n    keys: np.ndarray\n    drift_indices: list = field(default_factory=list)\n    burst_indices: list = field(default_factory=list)\n\n\ndef make_zipf_drift_trace(\n    n_requests: int,\n    key_space: int,\n    alpha: float,\n    n_drift_events: int,\n    drift_magnitude: float,\n    burst_prob: float,\n    seed: int,\n) -> TraceResult:\n    \"\"\"Zipf(alpha) popularity over `key_space` keys, with periodic hot-key\n    identity churn (drift) and occasional short bursts on a previously cold key.\n\n    Popularity SHAPE is held fixed (same Zipf exponent throughout); what\n    drifts is WHICH keys occupy the popular ranks, which is the regime a\n    per-key decay mechanism is meant to adapt to faster than a globally\n    reset sketch.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    ranks = np.arange(1, key_space + 1, dtype=np.float64)\n    probs = ranks ** (-alpha)\n    probs /= probs.sum()\n    rank_to_key = np.arange(key_space, dtype=np.int64)  # identity mapping initially\n\n    n_segments = n_drift_events + 1\n    seg_len = n_requests // n_segments\n    trace = np.empty(n_requests, dtype=np.int64)\n    drift_indices: list = []\n    burst_indices: list = []\n\n    top_frac_for_drift = max(1, int(round(drift_magnitude * key_space)))\n    burst_len = 200\n\n    pos = 0\n    for seg in range(n_segments):\n        this_len = seg_len if seg < n_segments - 1 else (n_requests - pos)\n        if this_len <= 0:\n            continue\n        rank_idx = rng.choice(key_space, size=this_len, p=probs)\n        seg_keys = rank_to_key[rank_idx]\n\n        if burst_prob > 0 and rng.random() < burst_prob and this_len > burst_len + 1:\n            # a cold key (bottom half of the rank distribution) bursts for a\n            # short contiguous window inside this segment\n            cold_rank = int(rng.integers(key_space // 2, key_space))\n            burst_key = int(rank_to_key[cold_rank])\n            start = int(rng.integers(0, this_len - burst_len))\n            seg_keys[start : start + burst_len] = burst_key\n            burst_indices.append(pos + start)\n\n        trace[pos : pos + this_len] = seg_keys\n        pos += this_len\n\n        if seg < n_segments - 1:\n            # drift: the top-`top_frac_for_drift` popular ranks get reassigned\n            # to a fresh random sample of key identities (old hot keys go\n            # cold, formerly-cold keys become hot).\n            top_indices = np.arange(top_frac_for_drift)\n            rank_to_key[top_indices] = rng.choice(key_space, size=top_frac_for_drift, replace=False)\n            drift_indices.append(pos)\n\n    return TraceResult(keys=trace, drift_indices=drift_indices, burst_indices=burst_indices)\n\n\nROLLING_WINDOW = 3000\nRECOVERY_LOOKAHEAD = 30000\nRECOVERY_TARGET_FRAC = 0.9\n\n\ndef _rolling_hit_ratio_fast(hit_bits: np.ndarray, window: int) -> np.ndarray:\n    \"\"\"O(n) rolling mean via cumulative sums (equivalent to the reference loop above).\"\"\"\n    n = len(hit_bits)\n    csum = np.cumsum(np.insert(hit_bits.astype(np.float64), 0, 0.0))\n    idx = np.arange(n)\n    lo = np.maximum(0, idx - window + 1)\n    counts = idx - lo + 1\n    return (csum[idx + 1] - csum[lo]) / counts\n\n\ndef run_trace(trace: np.ndarray, cache_capacity: int, estimator, window_admission_frac: float = 0.01) -> dict:\n    cache = WindowTinyLFUCache(cache_capacity, estimator, window_frac=window_admission_frac)\n    n = len(trace)\n    hit_bits = np.empty(n, dtype=np.uint8)\n    for i in range(n):\n        hit_bits[i] = 1 if cache.access(int(trace[i])) else 0\n    final_hit_ratio = float(hit_bits.mean())\n    rolling = _rolling_hit_ratio_fast(hit_bits, ROLLING_WINDOW)\n    return {\n        \"final_hit_ratio\": final_hit_ratio,\n        \"rolling_hit_ratio\": rolling,  # kept in-process only; summarized before JSON export\n        \"memory_bytes\": cache.memory_bytes(),\n    }\n\n\ndef compute_recovery_times(rolling: np.ndarray, drift_indices: list, lookahead: int = RECOVERY_LOOKAHEAD) -> list:\n    \"\"\"For each drift point, time until rolling hit ratio climbs back to\n    `RECOVERY_TARGET_FRAC` of the way from the post-drift trough back to the\n    pre-drift plateau. Returns `lookahead` (censored, logged) if it never does.\n    \"\"\"\n    n = len(rolling)\n    results = []\n    for d in drift_indices:\n        pre_lo, pre_hi = max(0, d - ROLLING_WINDOW), d\n        if pre_hi <= pre_lo:\n            continue\n        plateau = float(np.mean(rolling[pre_lo:pre_hi]))\n        search_lo = d + ROLLING_WINDOW\n        post_hi = min(n, d + lookahead)\n        if post_hi <= search_lo:\n            continue\n        window = rolling[search_lo:post_hi]\n        trough = float(np.min(window))\n        target = trough + RECOVERY_TARGET_FRAC * (plateau - trough)\n        recovered_offsets = np.where(window >= target)[0]\n        if len(recovered_offsets) == 0:\n            results.append({\"drift_index\": int(d), \"recovery_time\": lookahead, \"censored\": True})\n        else:\n            results.append(\n                {\"drift_index\": int(d), \"recovery_time\": int(recovered_offsets[0]) + ROLLING_WINDOW, \"censored\": False}\n            )\n    return results\n\n\ndef estimator_tier_diagnostics(estimator) -> Optional[dict]:\n    if isinstance(estimator, PerKeyDecayFrequencyEstimator):\n        total = max(1, sum(estimator.tier_assignment_counts))\n        return {\n            label: round(cnt / total, 4)\n            for (_, label), cnt in zip(estimator.TIERS, estimator.tier_assignment_counts)\n        }\n    return None\n\n\nprint(\"iter1 base simulator classes loaded\")

## Part A: short global-reset multiplier sweep vs the proposed per-key-decay estimator\n\nIter1 tuned the baseline's global-reset sample-size multiplier over `{4, 8, 16, 32}` and always picked the largest at the win-corner cell (`ratio=0.01, alpha=1.2`). This part asks the sharper question: can an even **shorter** reset period (`{1, 2, 4}`x cache capacity) close the recovery-time gap with the proposed per-key-decay estimator, without any per-key machinery at all?\n\nFor each of the 4 drift scenarios and each short-reset multiplier, `run_short_reset_cell` generates a synthetic Zipf-drift trace, runs the baseline estimator over it, and measures recovery time after each drift event.

In [ ]:
def load_iter1_proposed_results() -> dict:\n    \"\"\"Reads iter1's already-computed proposed-estimator (per-key decay) results\n    for every drift scenario at the win-corner cell, keyed by scenario name.\n    In the original script this comes from iter1's full_method_out.json; here it\n    is shipped pre-extracted in mini_demo_data.json (same values, not rerun).\n    \"\"\"\n    by_scenario: dict = {}\n    for r in iter1_proposed_recovery_times:\n        by_scenario.setdefault(r[\"scenario\"], []).append(r)\n    result = {}\n    for scenario, rows in by_scenario.items():\n        vals = [r[\"proposed_mean_recovery_time\"] for r in rows if r[\"proposed_mean_recovery_time\"] is not None]\n        result[scenario] = {\"per_seed\": rows, \"mean_across_seeds\": float(np.mean(vals)) if vals else None}\n    logger.info(f\"Loaded iter1 proposed-estimator recovery times for {len(result)} scenarios\")\n    return result\n\n\ndef run_short_reset_cell(scenario: dict, multiplier: int, seed: int) -> dict:\n    tr = make_zipf_drift_trace(\n        N_REQUESTS_MAIN,\n        KEY_SPACE,\n        ALPHA,\n        n_drift_events=scenario[\"n_drift_events\"],\n        drift_magnitude=scenario[\"drift_magnitude\"],\n        burst_prob=BURST_PROB,\n        seed=seed,\n    )\n    est = GlobalResetFrequencyEstimator(CACHE_CAPACITY, multiplier, seed=seed * 7 + 1)\n    res = run_trace(tr.keys, CACHE_CAPACITY, est)\n    recovery = compute_recovery_times(res[\"rolling_hit_ratio\"], tr.drift_indices, lookahead=RECOVERY_LOOKAHEAD_MAIN)\n    tail_start = int(0.85 * N_REQUESTS_MAIN)\n    steady = float(np.mean(res[\"rolling_hit_ratio\"][tail_start:]))\n    vals = [r[\"recovery_time\"] for r in recovery]\n    mean_recovery = float(np.mean(vals)) if vals else None\n    n_censored = sum(1 for r in recovery if r[\"censored\"])\n    return {\n        \"multiplier\": multiplier,\n        \"sample_size_W\": multiplier * CACHE_CAPACITY,\n        \"seed\": seed,\n        \"final_hit_ratio\": res[\"final_hit_ratio\"],\n        \"steady_state_hit_ratio\": steady,\n        \"memory_bytes\": res[\"memory_bytes\"],\n        \"mean_recovery_time\": mean_recovery,\n        \"n_drift_events\": len(tr.drift_indices),\n        \"n_censored_recovery_events\": n_censored,\n        \"recovery_events\": recovery,\n    }

In [ ]:
def run_part_a() -> dict:\n    logger.info(\n        f\"Part A: short-multiplier sweep at ratio={RATIO}, alpha={ALPHA} \"\n        f\"(cache_capacity={CACHE_CAPACITY}), multipliers={SHORT_MULTIPLIERS}, \"\n        f\"scenarios={len(DRIFT_SCENARIOS)}, seeds={SEEDS}\"\n    )\n    iter1_proposed = load_iter1_proposed_results()\n\n    per_run = []\n    by_scenario_mult: dict = {}\n    for scenario in DRIFT_SCENARIOS:\n        for mult in SHORT_MULTIPLIERS:\n            for seed in SEEDS:\n                t0 = time.time()\n                run = run_short_reset_cell(scenario, mult, seed)\n                run[\"scenario\"] = scenario[\"name\"]\n                per_run.append(run)\n                by_scenario_mult.setdefault((scenario[\"name\"], mult), []).append(run)\n                logger.info(\n                    f\"Part A: scenario={scenario['name']} mult={mult} seed={seed} \"\n                    f\"steady_hr={run['steady_state_hit_ratio']:.4f} \"\n                    f\"mean_recovery={run['mean_recovery_time']} \"\n                    f\"censored={run['n_censored_recovery_events']}/{run['n_drift_events']} \"\n                    f\"({time.time()-t0:.1f}s)\"\n                )\n\n    # Aggregate per (scenario, multiplier) across seeds\n    aggregated = []\n    for (scenario_name, mult), runs in by_scenario_mult.items():\n        rec_vals = [r[\"mean_recovery_time\"] for r in runs if r[\"mean_recovery_time\"] is not None]\n        hr_vals = [r[\"steady_state_hit_ratio\"] for r in runs]\n        collapse_flags = [r[\"n_censored_recovery_events\"] == r[\"n_drift_events\"] and r[\"n_drift_events\"] > 0 for r in runs]\n        aggregated.append(\n            {\n                \"scenario\": scenario_name,\n                \"multiplier\": mult,\n                \"sample_size_W\": mult * CACHE_CAPACITY,\n                \"n_seeds\": len(runs),\n                \"mean_recovery_time\": float(np.mean(rec_vals)) if rec_vals else None,\n                \"mean_steady_state_hit_ratio\": float(np.mean(hr_vals)),\n                \"fully_censored_seeds\": int(sum(collapse_flags)),\n                \"degenerate_admission_suspected\": bool(\n                    np.mean(hr_vals) < 0.5 or sum(collapse_flags) == len(runs)\n                ),\n            }\n        )\n\n    # Head-to-head: for each scenario, find the best (lowest mean recovery) short-reset\n    # multiplier and compare against iter1's already-computed proposed-estimator result.\n    head_to_head = []\n    for scenario in DRIFT_SCENARIOS:\n        name = scenario[\"name\"]\n        candidates = [a for a in aggregated if a[\"scenario\"] == name and a[\"mean_recovery_time\"] is not None]\n        if not candidates:\n            logger.warning(f\"Part A: scenario={name} has no valid (non-fully-censored) short-reset arm; skipping head-to-head\")\n            continue\n        best = min(candidates, key=lambda a: a[\"mean_recovery_time\"])\n        proposed_mean = iter1_proposed[name][\"mean_across_seeds\"] if name in iter1_proposed else None\n        if proposed_mean is None or best[\"mean_recovery_time\"] is None:\n            gap_pct = None\n        else:\n            gap_pct = 100.0 * (best[\"mean_recovery_time\"] - proposed_mean) / best[\"mean_recovery_time\"]\n        head_to_head.append(\n            {\n                \"scenario\": name,\n                \"best_short_reset_multiplier\": best[\"multiplier\"],\n                \"best_short_reset_mean_recovery_time\": best[\"mean_recovery_time\"],\n                \"best_short_reset_steady_state_hit_ratio\": best[\"mean_steady_state_hit_ratio\"],\n                \"proposed_estimator_mean_recovery_time_iter1\": proposed_mean,\n                \"proposed_still_faster_pct\": gap_pct,\n                \"interpretation\": (\n                    \"proposed per-key-decay estimator STILL recovers faster than the best \"\n                    \"short-reset global baseline -- short reset does not substitute for the mechanism\"\n                    if (gap_pct is not None and gap_pct > 0)\n                    else \"short-reset global baseline matches or beats the proposed estimator at this \"\n                    \"cell -- this DISCONFIRMS the necessity of per-key decay for this scenario\"\n                    if gap_pct is not None\n                    else \"comparison unavailable (missing data on one side)\"\n                ),\n            }\n        )\n        logger.info(\n            f\"Part A head-to-head [{name}]: best_short_reset(mult={best['multiplier']})=\"\n            f\"{best['mean_recovery_time']}, proposed(iter1)={proposed_mean}, \"\n            f\"proposed_still_faster_pct={gap_pct}\"\n        )\n\n    n_wins_for_proposed = sum(1 for h in head_to_head if h[\"proposed_still_faster_pct\"] is not None and h[\"proposed_still_faster_pct\"] > 0)\n    return {\n        \"per_run\": per_run,\n        \"aggregated_by_scenario_multiplier\": aggregated,\n        \"head_to_head_vs_iter1_proposed\": head_to_head,\n        \"summary\": {\n            \"n_scenarios_with_head_to_head\": len(head_to_head),\n            \"n_scenarios_proposed_still_wins\": n_wins_for_proposed,\n            \"fraction_scenarios_proposed_still_wins\": (\n                n_wins_for_proposed / len(head_to_head) if head_to_head else None\n            ),\n            \"any_degenerate_admission_observed\": any(a[\"degenerate_admission_suspected\"] for a in aggregated),\n        },\n    }\n\n\nt_a0 = time.time()\npart_a = run_part_a()\ngc.collect()\nprint(f\"Part A done in {time.time()-t_a0:.1f}s\")

## Part B: real Twitter production trace replay\n\nThis part replays both estimators end-to-end over a real sample of the Twitter `cache-trace` cluster026 (string keys are mapped to dense int ids for the shared sketch/SLRU code, exactly as the original does), tunes the baseline's multiplier fresh on the real trace itself, and then runs the unsupervised JS-divergence changepoint detector -- first validated against known drift events on a synthetic trace, then applied to the (unlabeled) real trace sample.

In [ ]:
def load_real_trace_keys() -> tuple:\n    \"\"\"Maps string keys from the real trace sample to dense int ids (required by\n    the Count-Min sketch / SLRU implementation, which is keyed on ints), and\n    returns (int_key_array, ordered_string_keys, request_types).\"\"\"\n    key_to_id: dict = {}\n    int_keys = np.empty(len(real_trace_sample), dtype=np.int64)\n    string_keys = []\n    request_types = []\n    for i, row in enumerate(real_trace_sample):\n        k = row[\"key\"]\n        string_keys.append(k)\n        request_types.append(row.get(\"request_type\", \"unknown\"))\n        idx = key_to_id.get(k)\n        if idx is None:\n            idx = len(key_to_id)\n            key_to_id[k] = idx\n        int_keys[i] = idx\n\n    rt_counts = Counter(request_types)\n    logger.info(\n        f\"Real trace sample: {len(real_trace_sample)} requests, {len(key_to_id)} distinct keys, \"\n        f\"request_type breakdown={dict(rt_counts)}\"\n    )\n    return int_keys, string_keys, request_types\n\n\ndef tune_real_trace_multiplier(int_keys: np.ndarray, cache_capacity: int) -> tuple:\n    \"\"\"Single-pass tuning: replays the trace once per candidate multiplier and\n    picks the multiplier with the best final hit ratio.\"\"\"\n    sweep = {}\n    best_mult, best_hr = SAMPLE_MULTIPLIERS[0], -1.0\n    for mult in SAMPLE_MULTIPLIERS:\n        est = GlobalResetFrequencyEstimator(cache_capacity, mult, seed=42)\n        res = run_trace(int_keys, cache_capacity, est)\n        sweep[mult] = res[\"final_hit_ratio\"]\n        if res[\"final_hit_ratio\"] > best_hr:\n            best_hr, best_mult = res[\"final_hit_ratio\"], mult\n    logger.info(f\"Real-trace multiplier tuning sweep: {sweep} -> chosen={best_mult}\")\n    return best_mult, sweep\n\n\ndef run_real_trace_replay(int_keys: np.ndarray, cache_capacity: int) -> dict:\n    best_mult, tuning_sweep = tune_real_trace_multiplier(int_keys, cache_capacity)\n\n    results = {}\n    for name, estimator in [\n        (\"baseline_w_tinylfu\", GlobalResetFrequencyEstimator(cache_capacity, best_mult, seed=101)),\n        (\n            \"per_key_decay\",\n            PerKeyDecayFrequencyEstimator(\n                cache_capacity, shadow_queue_capacity=SHADOW_QUEUE_MULT * cache_capacity, seed=102\n            ),\n        ),\n    ]:\n        t0 = time.time()\n        res = run_trace(int_keys, cache_capacity, estimator)\n        n = len(int_keys)\n        results[name] = {\n            \"final_hit_ratio\": res[\"final_hit_ratio\"],\n            \"steady_state_hit_ratio\": float(np.mean(res[\"rolling_hit_ratio\"][int(0.5 * n):])),\n            \"memory_bytes\": res[\"memory_bytes\"],\n            \"memory_bytes_per_cache_slot\": res[\"memory_bytes\"] / cache_capacity,\n            \"rolling_hit_ratio\": res[\"rolling_hit_ratio\"],  # kept in-process for changepoint recovery calc\n            \"tier_assignment_fractions\": estimator_tier_diagnostics(estimator),\n            \"runtime_seconds\": time.time() - t0,\n        }\n        logger.info(\n            f\"Real trace [{name}]: final_hr={res['final_hit_ratio']:.4f}, \"\n            f\"memory_bytes={res['memory_bytes']}, runtime={time.time()-t0:.1f}s\"\n        )\n    results[\"_meta\"] = {\"chosen_baseline_multiplier\": best_mult, \"tuning_sweep\": tuning_sweep, \"n_requests\": len(int_keys)}\n    return results\n\n\nint_keys, string_keys, request_types = load_real_trace_keys()\nn_distinct = int(int_keys.max()) + 1\nreal_cache_capacity = max(10, int(round(RATIO * n_distinct)))\nlogger.info(f\"Real trace: {n_distinct} distinct keys -> matched cache_capacity={real_cache_capacity} (ratio={RATIO})\")\n\nreplay = run_real_trace_replay(int_keys, real_cache_capacity)

### Unsupervised JS-divergence changepoint detector\n\nA rolling-window Jensen-Shannon divergence over the top-K key-identity frequency distribution flags candidate drift points. It is first validated against **known** drift events on a synthetic trace (recall/precision against ground truth) before being applied to the (unlabeled) real trace sample -- an untrustworthy detector is caught before being trusted on real data.

In [ ]:
def _key_freq_distribution(keys_window, top_k: int) -> dict:\n    counts = Counter(keys_window)\n    total = sum(counts.values())\n    top = counts.most_common(top_k)\n    dist = {k: c / total for k, c in top}\n    return dist\n\n\ndef _js_divergence(p: dict, q: dict) -> float:\n    keys = set(p) | set(q)\n    if not keys:\n        return 0.0\n    p_arr = np.array([p.get(k, 0.0) for k in keys])\n    q_arr = np.array([q.get(k, 0.0) for k in keys])\n    p_arr = p_arr / max(p_arr.sum(), 1e-12)\n    q_arr = q_arr / max(q_arr.sum(), 1e-12)\n    m = 0.5 * (p_arr + q_arr)\n\n    def _kl(a, b):\n        mask = a > 0\n        return float(np.sum(a[mask] * np.log2(a[mask] / np.maximum(b[mask], 1e-12))))\n\n    return 0.5 * _kl(p_arr, m) + 0.5 * _kl(q_arr, m)\n\n\ndef detect_changepoints(\n    keys: np.ndarray, window: int = CP_WINDOW, stride: int = CP_STRIDE, top_k: int = CP_TOP_K, percentile: float = CP_PERCENTILE\n) -> tuple:\n    n = len(keys)\n    starts = list(range(0, n - window, stride))\n    if len(starts) < 2:\n        return [], [], 0.0\n    dists = [_key_freq_distribution(keys[s : s + window], top_k) for s in starts]\n    js_scores = [_js_divergence(dists[i], dists[i + 1]) for i in range(len(dists) - 1)]\n    if not js_scores:\n        return [], [], 0.0\n    threshold = float(np.percentile(js_scores, percentile))\n    changepoints = [starts[i + 1] for i, s in enumerate(js_scores) if s > threshold]\n    return changepoints, js_scores, threshold\n\n\ndef validate_changepoint_detector_on_synthetic() -> dict:\n    \"\"\"Runs the SAME detector on a synthetic trace with KNOWN injected drift\n    events, and reports recall/precision against ground truth (with a generous\n    tolerance window) before trusting it on the unlabeled real trace.\"\"\"\n    tr = make_zipf_drift_trace(\n        N_REQUESTS_MAIN, KEY_SPACE, ALPHA, n_drift_events=8, drift_magnitude=0.2, burst_prob=0.5, seed=777\n    )\n    cps, js_scores, threshold = detect_changepoints(tr.keys)\n    true_events = tr.drift_indices\n    tolerance = CP_WINDOW + CP_STRIDE\n    matched_true = sum(1 for te in true_events if any(abs(te - cp) <= tolerance for cp in cps))\n    recall = matched_true / len(true_events) if true_events else None\n    matched_detected = sum(1 for cp in cps if any(abs(te - cp) <= tolerance for te in true_events))\n    precision = matched_detected / len(cps) if cps else None\n    result = {\n        \"n_true_drift_events\": len(true_events),\n        \"n_detected_changepoints\": len(cps),\n        \"tolerance_requests\": tolerance,\n        \"recall\": recall,\n        \"precision\": precision,\n        \"threshold\": threshold,\n        \"verdict\": (\n            \"DETECTOR_VALIDATED_ON_SYNTHETIC\"\n            if (recall is not None and recall > 0.3)\n            else \"DETECTOR_LOW_RECALL_TREAT_REAL_TRACE_CHANGEPOINTS_AS_WEAK_SIGNAL\"\n        ),\n    }\n    logger.info(f\"Changepoint detector synthetic validation: {result}\")\n    return result\n\n\nvalidation = validate_changepoint_detector_on_synthetic()\n\nlogger.info(\"Running changepoint detection over the real trace sample's per-key request stream\")\ncps, js_scores, threshold = detect_changepoints(int_keys)\npercentile_used = CP_PERCENTILE\nrelaxation_log = []\nif len(cps) == 0:\n    for p in (90.0, 85.0):\n        cps, js_scores, threshold = detect_changepoints(int_keys, percentile=p)\n        relaxation_log.append({\"percentile_tried\": p, \"n_changepoints\": len(cps)})\n        percentile_used = p\n        if cps:\n            break\nlogger.info(f\"Detected {len(cps)} candidate changepoints at percentile={percentile_used} (threshold={threshold:.5f})\")

## Results\n\nSummary tables and plots for both parts: Part A's head-to-head recovery-time comparison (best short-reset baseline vs. iter1's proposed per-key-decay estimator) across drift scenarios, and Part B's real-trace hit-ratio / memory comparison plus the changepoint detector's synthetic validation.

In [ ]:
print(\"=\" * 78)\nprint(\"PART A: best short-reset baseline vs. iter1 proposed per-key-decay estimator\")\nprint(\"=\" * 78)\nprint(f\"{'scenario':<20}{'best_mult':>10}{'best_recovery':>16}{'proposed_recovery':>20}{'proposed_faster_%':>20}\")\nfor h in part_a[\"head_to_head_vs_iter1_proposed\"]:\n    print(\n        f\"{h['scenario']:<20}{h['best_short_reset_multiplier']:>10}\"\n        f\"{h['best_short_reset_mean_recovery_time']:>16.1f}\"\n        f\"{(h['proposed_estimator_mean_recovery_time_iter1'] or float('nan')):>20.1f}\"\n        f\"{(h['proposed_still_faster_pct'] or float('nan')):>20.1f}\"\n    )\nprint(f\"\\nProposed estimator still wins in {part_a['summary']['n_scenarios_proposed_still_wins']}/\"\n      f\"{part_a['summary']['n_scenarios_with_head_to_head']} scenarios at this (much smaller) demo scale.\")\n\nprint(\"\\n\" + \"=\" * 78)\nprint(\"PART B: real Twitter trace sample replay\")\nprint(\"=\" * 78)\nfor name in [\"baseline_w_tinylfu\", \"per_key_decay\"]:\n    r = replay[name]\n    print(f\"{name:<20} final_hit_ratio={r['final_hit_ratio']:.4f}  memory_bytes={r['memory_bytes']:.0f}\")\nprint(f\"\\nChangepoint detector synthetic validation: recall={validation['recall']}, \"\n      f\"precision={validation['precision']}, verdict={validation['verdict']}\")\nprint(f\"Candidate changepoints detected on real trace sample: {len(cps)} (percentile={percentile_used})\")\n\n# --- Plots ---\nfig, axes = plt.subplots(1, 3, figsize=(16, 4.5))\n\n# Panel 1: Part A recovery-time head-to-head\nscenarios = [h[\"scenario\"] for h in part_a[\"head_to_head_vs_iter1_proposed\"]]\nbest_short = [h[\"best_short_reset_mean_recovery_time\"] for h in part_a[\"head_to_head_vs_iter1_proposed\"]]\nproposed = [h[\"proposed_estimator_mean_recovery_time_iter1\"] or 0 for h in part_a[\"head_to_head_vs_iter1_proposed\"]]\nx = np.arange(len(scenarios))\nwidth = 0.35\naxes[0].bar(x - width / 2, best_short, width, label=\"best short-reset baseline\")\naxes[0].bar(x + width / 2, proposed, width, label=\"proposed per-key-decay (iter1)\")\naxes[0].set_xticks(x)\naxes[0].set_xticklabels(scenarios, rotation=30, ha=\"right\", fontsize=8)\naxes[0].set_ylabel(\"mean recovery time (requests)\")\naxes[0].set_title(\"Part A: recovery time by scenario\")\naxes[0].legend(fontsize=8)\n\n# Panel 2: Part B hit ratio / memory comparison\nnames = [\"baseline_w_tinylfu\", \"per_key_decay\"]\nhrs = [replay[n][\"final_hit_ratio\"] for n in names]\nmems = [replay[n][\"memory_bytes\"] for n in names]\nax2 = axes[1]\nax2.bar(names, hrs, color=[\"C0\", \"C1\"])\nax2.set_ylabel(\"final hit ratio\")\nax2.set_title(\"Part B: real trace hit ratio\")\nax2.tick_params(axis=\"x\", labelrotation=15)\n\nax3 = axes[2]\nax3.bar(names, mems, color=[\"C0\", \"C1\"])\nax3.set_ylabel(\"memory bytes\")\nax3.set_title(\"Part B: real trace memory footprint\")\nax3.tick_params(axis=\"x\", labelrotation=15)\n\nplt.tight_layout()\nplt.show()